In [1]:
#Imports
import psycopg2
from psycopg2 import sql

In [2]:
import os
from pathlib import Path

# Connection settings come from the environment (see .env.example at the repo root).
PG = dict(
    host=os.environ.get("POSTGRES_HOST", "localhost"),
    port=os.environ.get("POSTGRES_PORT", "5432"),
    dbname=os.environ.get("PEDIATRIC_DB", "pediatric"),
    user=os.environ.get("POSTGRES_USER", "postgres"),
    password=os.environ.get("POSTGRES_PASSWORD"),
)

#Koneksi postgre
try:
    #Konfigurasi koneksi
    conn = psycopg2.connect(**PG)
    print("Koneksi ke database pediatric berhasil!")
    #Membuat cursor
    cur = conn.cursor()
    print("Cursor berhasil dibuat!")
except psycopg2.Error as e:
    print("Gagal terhubung ke database:", e)

Koneksi ke database pediatric berhasil!
Cursor berhasil dibuat!


In [3]:
#Membuat extension UUID
cur.execute("CREATE EXTENSION IF NOT EXISTS \"uuid-ossp\";")
conn.commit()

In [4]:
#Dimensi Biodata Pasien (SCD Type 2)
cur.execute("""
    CREATE TABLE IF NOT EXISTS biodata_pasien (
        pasien_id UUID PRIMARY KEY DEFAULT uuid_generate_v4(),
        nama VARCHAR(255),
        umur INT,
        kelamin VARCHAR(10),
        alamat TEXT,
        no_tlp VARCHAR(20),
        start_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        end_date TIMESTAMP,
        active BOOLEAN DEFAULT TRUE
    );
""")

In [5]:
#Dimensi Pengkajian Pasien (SCD Type 2)
cur.execute("""
    CREATE TABLE IF NOT EXISTS pengkajian_pasien (
        pengkajian_id UUID PRIMARY KEY DEFAULT uuid_generate_v4(),
        pasien_id UUID REFERENCES biodata_pasien(pasien_id),
        tinggi DECIMAL(5,2),
        berat DECIMAL(5,2),
        tekanan VARCHAR(20),
        suhu DECIMAL(4,2),
        tinggi_bapak DECIMAL(5,2),
        tinggi_ibu DECIMAL(5,2),
        start_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        end_date TIMESTAMP,
        active BOOLEAN DEFAULT TRUE
    );
""")

In [6]:
#Dimensi Laboratorium
cur.execute("""
    CREATE TABLE IF NOT EXISTS laboratorium (
        laboratorium_id UUID PRIMARY KEY DEFAULT uuid_generate_v4(),
        jenis_tes VARCHAR(255),
        hasil_tes TEXT,
        tanggal_tes DATE
    );
""")

In [7]:
#Dimensi Konsultasi
cur.execute("""
    CREATE TABLE IF NOT EXISTS konsultasi (
        konsultasi_id UUID PRIMARY KEY DEFAULT uuid_generate_v4(),
        nama_dokter VARCHAR(255),
        diagnosis TEXT,
        treatment TEXT,
        deskripsi TEXT
    );
""")

In [8]:
#Dimensi Farmasi
cur.execute("""
    CREATE TABLE IF NOT EXISTS farmasi (
        farmasi_id UUID PRIMARY KEY DEFAULT uuid_generate_v4(),
        nama_obat VARCHAR(255),
        dosis VARCHAR(50),
        frekuensi VARCHAR(50)
    );
""")

In [9]:
#Dimensi Fakta
cur.execute("""
    CREATE TABLE IF NOT EXISTS fact_checkup (
        pasien_id UUID REFERENCES biodata_pasien(pasien_id),
        pengkajian_id UUID REFERENCES pengkajian_pasien(pengkajian_id),
        laboratorium_id UUID REFERENCES laboratorium(laboratorium_id),
        konsultasi_id UUID REFERENCES konsultasi(konsultasi_id),
        farmasi_id UUID REFERENCES farmasi(farmasi_id),
        tanggal_checkup TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        biaya DECIMAL(10,2),
        PRIMARY KEY (pasien_id, pengkajian_id, laboratorium_id, konsultasi_id, farmasi_id)
    );
""")

conn.commit()
print("Schema Star berhasil dibuat!")

Schema Star berhasil dibuat!
